# ASMSA v2 — Live metadynamics monitor

← [Metadynamics setup and run](06-md.ipynb) &nbsp;|&nbsp; [Analysis](07-analysis.ipynb) →

Open this notebook in a **second Jupyter kernel on the same host** as the production
run in [06-md](06-md.ipynb). The host and project directory must be shared, because
this monitor reads the run's `COLVAR`, `HILLS`, trajectory metadata, and GROMACS log.

This notebook is **read-only with respect to the simulation**: it does not start,
stop, signal, or rewrite GROMACS or PLUMED files. Interrupting the monitor cell stops
only its refresh loop. The dashboard reports file changes and, when available, the
`asmsa_md` tmux session; these are operational clues, not proof of convergence.

The latent-space plot can show the current CV values against the values seen in the
training trajectory. Metadynamics is intended to explore new regions, so values beyond
the training trajectory's observed minimum/maximum are **not by themselves evidence of
a bad CV or a PBC error**. To verify the exported CV, compare Python and PLUMED on the
same frames using the bias-free `plumed driver` check in [05-plumed](05-plumed.ipynb).

The monitor reads only complete rows appended to `COLVAR` and `HILLS`. It checks the
size and update time of `md.xtc` but never opens the trajectory while GROMACS may be
writing it. A flat trace, a stopped file, or a tmux session alone is not a scientific
convergence test.

In [ ]:
from collections import deque
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display

import asmsa_v2 as asmsa
from asmsa_v2.viz import use_asmsa_style

use_asmsa_style()
project = asmsa.Project.load('run')
md_dir = project.dir('md')
print(f'Project root: {project.root.resolve()}')
print(f'Monitoring:   {md_dir.resolve()}')
print(f'Target run:   {project.production_ns:g} ns')

## Start the live view

1. Run the setup cell above to load the project and resolve its `04-md` directory.
2. Start production from [06-md](06-md.ipynb) on the same host. That notebook's
   persistent run uses the tmux session `asmsa_md`.
3. Run the next cell here. The dashboard refreshes every `REFRESH_SECONDS`.
4. Press **Interrupt** to stop only this dashboard loop. No stop signal is sent to
   the production process.

**How to interpret it:** COLVAR time is converted from the GROMACS/PLUMED time unit
(ps) to ns and compared with the configured production length. It is an estimate,
not a process-completion percentage; a resumed or extended simulation can exceed
100%. File timestamps and tmux are supporting clues only. If the files stop changing,
inspect the GROMACS log and the tmux session on the run host.

The live performance estimate uses the change in COLVAR simulation time divided by
the monitor's elapsed wall time. It reports both ns/h and ns/day, averaged over a
rolling window of up to 10 minutes; the first minute is used to collect data. This is
an observed throughput estimate, not a hardware benchmark or a convergence measure.
If COLVAR time does not advance, performance is shown as unavailable rather than
assuming that the simulation has stopped.

CV values outside the training trajectory's observed range may be the intended result
of enhanced exploration. This monitor reports their frequency in the retained data;
it does not label them as an export failure. Check Python-versus-PLUMED CV agreement
with `plumed driver` before production, and investigate discontinuities or other
independent evidence of a mapping/PBC problem.

The plots use COLVAR values and mark the first and latest finite COLVAR points. HILLS
is currently used for the deposited-hill count, not as a plotted trajectory.

In [ ]:
import re
import shutil
import subprocess

REFRESH_SECONDS = 15
PERFORMANCE_WINDOW_SECONDS = 10 * 60
PERFORMANCE_MIN_INTERVAL_SECONDS = 60
MAX_PLOT_POINTS = 100000  # retained recent rows for summaries and CV-range checks
MAX_RENDER_POINTS = 5000  # cap plotted samples so refreshes stay responsive
TMUX_SESSION = 'asmsa_md'

colvar_path = md_dir / 'COLVAR'
hills_path = md_dir / 'HILLS'
trajectory_path = md_dir / 'md.xtc'
log_path = md_dir / 'logs' / 'gmx-mdrun.log'
train_latent_path = project.dir('models') / f'latent-{project.model_name}.npy'

training_latent = None
training_min = training_max = None
if train_latent_path.exists():
    training_latent = np.load(train_latent_path, mmap_mode='r')
    if training_latent.ndim != 2 or training_latent.shape[1] < 2:
        raise ValueError(
            f'Expected at least two latent dimensions in {train_latent_path}; '
            f'got shape {training_latent.shape}'
        )
    if not np.isfinite(training_latent[:, :2]).all():
        raise ValueError(f'Training latent contains non-finite values: {train_latent_path}')
    training_min = np.min(training_latent[:, :2], axis=0)
    training_max = np.max(training_latent[:, :2], axis=0)
    print(f'Training latent reference: {train_latent_path} ({len(training_latent):,} points)')
else:
    print(f'No training latent found at {train_latent_path}; CV-space plot will show MD only.')


def new_reader(max_points):
    return {
        'identity': None,
        'offset': 0,
        'pending': b'',
        'fields': None,
        'rows': deque(maxlen=max_points),
        'first_finite_row': None,
        'count': 0,
    }


colvar_reader = new_reader(MAX_PLOT_POINTS)
hills_reader = new_reader(1)
performance_samples = deque()


def observed_performance(latest_ns):
    """Estimate throughput from COLVAR progress observed by this monitor."""
    now = time.monotonic()
    if performance_samples and latest_ns < performance_samples[-1][1]:
        performance_samples.clear()
    performance_samples.append((now, latest_ns))
    while (len(performance_samples) > 1
           and now - performance_samples[0][0] > PERFORMANCE_WINDOW_SECONDS):
        performance_samples.popleft()

    elapsed_s = performance_samples[-1][0] - performance_samples[0][0]
    if elapsed_s < PERFORMANCE_MIN_INTERVAL_SECONDS:
        return f'collecting data ({elapsed_s:.0f}/{PERFORMANCE_MIN_INTERVAL_SECONDS} s)'

    simulated_ns = performance_samples[-1][1] - performance_samples[0][1]
    if simulated_ns <= 0:
        return 'unavailable (COLVAR time has not advanced during the window)'

    ns_per_hour = simulated_ns / elapsed_s * 3600
    ns_per_day = ns_per_hour * 24
    return (f'{ns_per_hour:.3g} ns/h ({ns_per_day:.3g} ns/day), '
            f'rolling window {elapsed_s / 60:.1f} min')



def read_appended_rows(path, state):
    """Append complete PLUMED data rows, detecting file replacement/truncation."""
    if not path.exists():
        return state

    stat = path.stat()
    identity = (stat.st_dev, stat.st_ino)
    if state['identity'] != identity or stat.st_size < state['offset']:
        state.update(
            identity=identity,
            offset=0,
            pending=b'',
            fields=None,
            rows=deque(maxlen=state['rows'].maxlen),
            first_finite_row=None,
            count=0,
        )

    with path.open('rb') as stream:
        stream.seek(state['offset'])
        chunk = stream.read()
        state['offset'] += len(chunk)

    content = state['pending'] + chunk
    complete, state['pending'] = content.rsplit(b'\n', 1) if b'\n' in content else (b'', content)
    for raw_line in complete.splitlines():
        line = raw_line.decode('utf-8', errors='replace').strip()
        if not line:
            continue
        if line.startswith('#! FIELDS'):
            state['fields'] = line.split()[2:]
            continue
        if line.startswith('#') or line.startswith('@'):
            continue
        try:
            values = [float(value) for value in line.split()]
        except ValueError as exc:
            raise ValueError(f'Could not parse a data row in {path.name}: {line}') from exc
        if state['fields'] is not None and len(values) != len(state['fields']):
            raise ValueError(f'Unexpected column count in {path.name}: {line}')
        if state['first_finite_row'] is None and np.isfinite(values).all():
            state['first_finite_row'] = values
        state['rows'].append(values)
        state['count'] += 1
    return state


def file_status(path):
    if not path.exists():
        return 'not created yet'
    stat = path.stat()
    age = max(0, time.time() - stat.st_mtime)
    size_mb = stat.st_size / (1024 * 1024)
    fresh_after = max(120, 4 * REFRESH_SECONDS)
    freshness = (
        f'updated {age:.0f} s ago'
        if age < fresh_after
        else f'no update for {age / 60:.1f} min'
    )
    return f'{size_mb:.2f} MB; {freshness}'


def file_snapshot(paths):
    snapshot = {}
    for label, path in paths.items():
        if path.exists():
            stat = path.stat()
            snapshot[label] = (stat.st_dev, stat.st_ino, stat.st_size, stat.st_mtime_ns)
        else:
            snapshot[label] = None
    return snapshot


def tmux_status():
    """Report tmux evidence without treating session existence as proof of mdrun."""
    if shutil.which('tmux') is None:
        return 'tmux unavailable here; process state not confirmed'

    check = subprocess.run(
        ['tmux', 'has-session', '-t', TMUX_SESSION],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
        timeout=2,
    )
    if check.returncode != 0:
        return f'no {TMUX_SESSION!r} tmux session detected; run may be elsewhere, finished, or not started'

    panes = subprocess.run(
        ['tmux', 'list-panes', '-t', TMUX_SESSION, '-F', '#{pane_current_command}|#{pane_current_path}|#{pane_dead}'],
        capture_output=True,
        text=True,
        check=False,
        timeout=2,
    )
    if panes.returncode != 0:
        return f'{TMUX_SESSION!r} session exists; pane details unavailable'

    descriptions = []
    for line in panes.stdout.splitlines():
        command, _, rest = line.partition('|')
        pane_path, _, dead = rest.rpartition('|')
        if dead == '1':
            descriptions.append(f'pane exited (last command: {command or "unknown"})')
        elif re.search(r'gmx|mdrun', command, flags=re.IGNORECASE):
            same_dir = Path(pane_path).resolve() == md_dir.resolve() if pane_path else False
            suffix = '; working directory matches' if same_dir else '; working directory differs or is unknown'
            descriptions.append(f'GROMACS-like pane command: {command}{suffix}')
        else:
            descriptions.append(f'session exists; pane command is {command or "unknown"}')
    return '; '.join(descriptions) if descriptions else f'{TMUX_SESSION!r} session exists; no pane details'


def log_tail(path, lines=10):
    if not path.exists():
        return 'No GROMACS log found at the expected path yet.'
    with path.open('rb') as stream:
        stream.seek(max(0, path.stat().st_size - 8192))
        text = stream.read().decode('utf-8', errors='replace')
    return '\n'.join(text.splitlines()[-lines:]) or '(log is empty)'


def minmax_sample_indices(values, max_points):
    """Choose chronological plot indices while retaining each bucket's extrema."""
    values = np.asarray(values)
    n_rows, n_columns = values.shape
    if n_rows <= max_points:
        return np.arange(n_rows)

    n_buckets = max(1, max_points // max(2, 2 * n_columns))
    edges = np.linspace(0, n_rows, n_buckets + 1, dtype=int)
    selected = {0, n_rows - 1}
    for start, stop in zip(edges[:-1], edges[1:]):
        if stop <= start:
            continue
        block = values[start:stop]
        for column in range(n_columns):
            finite = np.flatnonzero(np.isfinite(block[:, column]))
            if finite.size:
                selected.add(start + int(finite[np.argmin(block[finite, column])]))
                selected.add(start + int(finite[np.argmax(block[finite, column])]))
    return np.fromiter(sorted(selected), dtype=np.int64)


watch_paths = {
    'COLVAR': colvar_path,
    'HILLS': hills_path,
    'trajectory': trajectory_path,
    'GROMACS log': log_path,
}
previous_snapshot = None


def render_dashboard():
    global previous_snapshot
    read_appended_rows(colvar_path, colvar_reader)
    read_appended_rows(hills_path, hills_reader)
    snapshot = file_snapshot(watch_paths)

    clear_output(wait=True)
    print(f'Live metadynamics monitor — {time.strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'Project: {project.root.resolve()}')
    print(f'Run monitor evidence: {tmux_status()}')
    for label, path in watch_paths.items():
        note = ' (size/update only; not opened)' if label == 'trajectory' else ''
        print(f'{label}: {file_status(path)}{note}')

    if previous_snapshot is None:
        print('File-change tracking: baseline captured; changes will be reported next refresh.')
    else:
        changed = [name for name in watch_paths if snapshot[name] != previous_snapshot[name]]
        print('Files changed since last refresh: ' + (', '.join(changed) if changed else 'none observed'))
    previous_snapshot = snapshot

    fields = colvar_reader['fields'] or []
    rows = list(colvar_reader['rows'])
    if not rows or not fields:
        print('\nWaiting for complete COLVAR data rows. The simulation may not have started yet.')
    else:
        data = np.asarray(rows, dtype=float)
        columns = {name: data[:, i] for i, name in enumerate(fields)}
        time_col = 'time' if 'time' in columns else fields[0]
        time_values = columns[time_col]
        valid_time = np.isfinite(time_values)
        if not valid_time.any():
            print(f'\nNo finite values found in the COLVAR time column {time_col!r}.')
            return

        latest_ns = float(time_values[valid_time][-1]) / 1000.0
        target_ns = float(project.production_ns)
        estimated_pct = 100 * latest_ns / target_ns if target_ns > 0 else float('nan')
        finite_rows = np.isfinite(data).all(axis=1)
        print(f'\nCOLVAR records read: {colvar_reader["count"]:,} '
              f'(retained {len(rows):,}; plot summaries use finite rows)')
        print(f'Latest COLVAR time: {latest_ns:.3f} ns; configured target: {target_ns:g} ns '
              f'({estimated_pct:.1f}% by time; not process completion)')
        print(f'Observed simulation performance: {observed_performance(latest_ns)}')
        if not finite_rows.all():
            print(f'WARNING: {int((~finite_rows).sum()):,} retained COLVAR rows contain non-finite values.')
        if hills_reader['count']:
            print(f'Deposited hills observed: {hills_reader["count"]:,}')

        bias_cols = [name for name in fields if name.endswith('.bias')]
        cv_cols = [name for name in fields if re.fullmatch(r'model\.node-\d+', name)]
        cv_cols.sort(key=lambda name: int(name.rsplit('-', 1)[1]))
        if not cv_cols:
            reserved = {time_col, *bias_cols}
            cv_cols = [name for name in fields if name not in reserved and not name.endswith('.rct')]
        cv_cols = cv_cols[:2]

        plot_data = data[finite_rows]
        plot_columns = {name: plot_data[:, i] for i, name in enumerate(fields)}
        plot_time_ns = plot_columns[time_col] / 1000.0
        first_endpoint = colvar_reader['first_finite_row']
        latest_endpoint = plot_data[-1] if len(plot_data) else None

        def mark_endpoints(ax, x_name, y_name, time_axis=False, show_labels=True):
            endpoints = (
                ('first COLVAR point', first_endpoint, 'o', 'forestgreen'),
                ('latest COLVAR point', latest_endpoint, 'X', 'darkorange'),
            )
            x_index = fields.index(time_col if time_axis else x_name)
            y_index = fields.index(y_name)
            for label, row, marker, color in endpoints:
                if row is None:
                    continue
                x_value, y_value = float(row[x_index]), float(row[y_index])
                if not np.isfinite([x_value, y_value]).all():
                    continue
                if time_axis:
                    x_value /= 1000.0
                ax.scatter(
                    [x_value], [y_value], marker=marker, s=115, c=color,
                    edgecolors='black', linewidths=1.0,
                    label=label if show_labels else '_nolegend_', zorder=5,
                )

        if training_min is not None and cv_cols == ['model.node-0', 'model.node-1'] and len(plot_data):
            cv_values = plot_data[:, [fields.index(name) for name in cv_cols]]
            outside = np.any((cv_values < training_min) | (cv_values > training_max), axis=1)
            n_outside = int(outside.sum())
            fraction = 100 * n_outside / len(cv_values)
            print(f'Training observed range: [{training_min[0]:.3g}, {training_max[0]:.3g}] x '
                  f'[{training_min[1]:.3g}, {training_max[1]:.3g}]')
            print(f'Retained live points outside those extrema: {n_outside:,}/{len(cv_values):,} '
                  f'({fraction:.1f}%)')
            print('Interpretation: new-range exploration can be expected in metadynamics; '
                  'this comparison alone does not diagnose a mapping/PBC error.')

        panels = 1 + bool(bias_cols) + (1 if len(cv_cols) >= 2 else 0)
        fig, axes = plt.subplots(
            panels, 1, figsize=(14, 3.8 * panels), dpi=120, squeeze=False
        )
        fig.set_facecolor('white')
        try:
            ax_index = 0
            ax = axes[ax_index, 0]
            trace_names = cv_cols + bias_cols
            if len(plot_data):
                trace_values = np.column_stack([plot_columns[name] for name in trace_names]) if trace_names else np.empty((len(plot_data), 0))
                sample = minmax_sample_indices(trace_values, MAX_RENDER_POINTS) if trace_names else np.arange(len(plot_data))
            else:
                sample = np.array([], dtype=np.int64)
            for i, name in enumerate(cv_cols):
                ax.plot(plot_time_ns[sample], plot_columns[name][sample], lw=1.2, label=name)
                mark_endpoints(ax, time_col, name, time_axis=True, show_labels=i == 0)
            ax.set_ylabel('CV value')
            ax.set_xlabel('time (ns; COLVAR ps converted to ns)')
            ax.set_title('Collective variables over simulation time', fontsize=13, pad=10)
            if cv_cols:
                ax.legend(loc='best', fontsize=10)
            else:
                ax.text(0.5, 0.5, 'No CV columns found in COLVAR', ha='center', va='center', transform=ax.transAxes)
            ax_index += 1

            if bias_cols:
                ax = axes[ax_index, 0]
                for i, name in enumerate(bias_cols):
                    ax.plot(plot_time_ns[sample], plot_columns[name][sample], lw=1.2, label=name)
                    mark_endpoints(ax, time_col, name, time_axis=True, show_labels=False)
                ax.set_ylabel('bias (kJ/mol)')
                ax.set_xlabel('time (ns)')
                ax.set_title('Metadynamics bias', fontsize=13, pad=10)
                ax.legend(loc='best', fontsize=10)
                ax_index += 1

            if len(cv_cols) >= 2:
                ax = axes[ax_index, 0]
                if training_latent is not None and cv_cols == ['model.node-0', 'model.node-1']:
                    train_step = max(1, len(training_latent) // MAX_RENDER_POINTS)
                    ax.scatter(training_latent[::train_step, 0], training_latent[::train_step, 1],
                               s=4, c='0.65', alpha=0.35, label='training trajectory', zorder=1)
                live_scatter = None
                if len(plot_data):
                    scatter_step = max(1, len(plot_data) // MAX_RENDER_POINTS)
                    scatter_rows = np.arange(0, len(plot_data), scatter_step)
                    if scatter_rows[-1] != len(plot_data) - 1:
                        scatter_rows = np.append(scatter_rows, len(plot_data) - 1)
                    live_scatter = ax.scatter(
                        plot_columns[cv_cols[0]][scatter_rows],
                        plot_columns[cv_cols[1]][scatter_rows],
                        c=plot_time_ns[scatter_rows], cmap='viridis',
                        s=12, alpha=0.75, label='metadynamics', zorder=2,
                    )
                mark_endpoints(ax, cv_cols[0], cv_cols[1], show_labels=False)
                if live_scatter is not None:
                    fig.colorbar(
                        live_scatter, ax=ax, pad=0.02, label='simulation time (ns)'
                    )
                ax.set_xlabel(cv_cols[0])
                ax.set_ylabel(cv_cols[1])
                ax.set_title('CV space explored during metadynamics', fontsize=13, pad=10)
                ax.legend(loc='best', fontsize=8, markerscale=1.2)

            for ax in axes[:, 0]:
                ax.grid(axis='both', color='0.85', linestyle='--', linewidth=0.7)
                ax.set_axisbelow(True)
                ax.tick_params(labelsize=10)
                ax.xaxis.label.set_size(11)
                ax.yaxis.label.set_size(11)
                ax.spines[['top', 'right']].set_visible(False)

            fig.tight_layout()
            display(fig)
        finally:
            plt.close(fig)

    print('\nRecent GROMACS log output:')
    print(log_tail(log_path))

In [ ]:
try:
    while True:
        render_dashboard()
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print('Monitoring stopped. This cell sent no stop signal to GROMACS.')